In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'

In [2]:
import pandas as pd
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig
)
import torch
import json
from tqdm import tqdm

/data/jfraile/ml/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_path='/data/jfraile/llms/granite-3.3-2b-instruct'
model_name='granite'

In [4]:
model= AutoModelForCausalLM.from_pretrained(model_path, device_map='auto', torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.91it/s]


In [6]:
def interact_with_model(prompt, max_tokens=500):
    pipeline1 = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    )

    # Parámetros de generación
    pipeline_kwargs = {
        "do_sample": True,
        "temperature": 0.75,
        "max_new_tokens": max_tokens,
        "return_full_text": False
    }
    chat=[{'role':'user', 'content':prompt}]
    final_prompt=tokenizer.apply_chat_template(chat, tokenize=False)
    pipeline_kwargs=pipeline_kwargs
    pipeline_kwargs['eos_token_id']=tokenizer.eos_token_id
    for out in pipeline1(final_prompt,**pipeline_kwargs):
        model_out=out['generated_text']
    return model_out




In [7]:
df_titles_desc=pd.read_json('/data/jfraile/Programs/datasets/SemEval_2025_T10_S2/titles_and_descriptions.json')
ids_with_title_desc=df_titles_desc.id.values

allowed_subnarratives=['URW1.1', 'URW1.2', 'URW1.OTH', 'URW2.1', 'URW2.2', 'URW2.3', 'URW2.4', 'URW2.5', 'URW2.6', 'URW2.7', 'URW2.8', 'URW2.OTH', 'URW3.1', 'URW3.2', 'URW3.3', 'URW3.OTH', 'URW4.1', 'URW4.2', 'URW4.3', 'URW4.4', 'URW4.5', 'URW4.OTH', 'URW5.1', 'URW5.2', 'URW5.3', 'URW5.OTH', 'URW6.1', 'URW6.2', 'URW6.3', 'URW6.OTH', 'URW7.1', 'URW7.2', 'URW7.3', 'URW7.4', 'URW7.5', 'URW7.6', 'URW7.OTH', 'URW8.1', 'URW8.2', 'URW8.OTH', 'URW9.1', 'URW9.2', 'URW9.OTH', 'URW10.1', 'URW10.2', 'URW10.3', 'URW10.4', 'URW10.OTH', 'URW11.OTH', 'CC1.1', 'CC1.2', 'CC1.3', 'CC1.OTH', 'CC2.1', 'CC2.2', 'CC2.3', 'CC2.4', 'CC2.OTH', 'CC3.1', 'CC3.2', 'CC3.OTH', 'CC4.1', 'CC4.2', 'CC4.3', 'CC4.4', 'CC4.5', 'CC4.6', 'CC4.7', 'CC4.8', 'CC4.OTH', 'CC5.1', 'CC5.2', 'CC5.3', 'CC5.4', 'CC5.OTH', 'CC6.1', 'CC6.2', 'CC6.3', 'CC6.OTH', 'CC7.1', 'CC7.2', 'CC7.3', 'CC7.4', 'CC7.OTH', 'CC8.1', 'CC8.2', 'CC8.OTH', 'CC9.1', 'CC9.2', 'CC9.3', 'CC9.4', 'CC9.OTH', 'CC10.1', 'CC10.2', 'CC10.OTH', 'OTH.OTH']

allowed_narratives=[f'URW{i}.' for i in range(1,12)] + [f'CC{i}.' for i in range(1,11)]

# Description from titles

In [8]:
def get_prompt_subnarr(narr, title_subnarr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a subnarrative about the narrative "{narr}". Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the subnarrative. Your description will have to help improve the identification of subnarratives in news.

Subnarrative title: {title_subnarr} 

Task: Generate a description of the subnarrative. 
    """
    return user_prompt

def get_prompt_narr(narr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a narrative. Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the narrative. Your description will have to help improve the identification of narratives in news.

Narrative title: {narr} 

Task: Generate a description of the narrative. 
    """
    return user_prompt

In [9]:
name=f'{model_name}_z_s_desc_only_title'
try:
    df_results=pd.read_json(f'/data/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}.json')
    df_results=df_results[df_results['description'].apply(lambda x: 'Error al interactuar con la API' not in x)]
except:    
    df_results=pd.DataFrame(columns=['id', 'description'])

for subnarr in tqdm(allowed_subnarratives):
    print('='*10)
    if 'OTH' in subnarr:
        continue
    try:
        t=df_results[df_results['id']==subnarr]['description'].iloc[0]
    except:
        t='Error al interactuar con la API'
    if 'Error al interactuar con la API' in t:
        narr=subnarr.split('.')[0]+'.'
        narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
        narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
        subnarr_title=df_titles_desc[df_titles_desc['id']==subnarr]['title'].iloc[0]
        subnarr_desc=df_titles_desc[df_titles_desc['id']==subnarr]['description'].iloc[0]
        prompt=get_prompt_subnarr(narr_title, subnarr_title)
        # print(prompt)
        result=interact_with_model(prompt)
        print(result)
        new_row = pd.DataFrame([{'id':subnarr, 'description': result}])
        df_results = pd.concat([df_results, new_row], ignore_index=True)

for narr in tqdm(allowed_narratives):
    print('='*10)
    try:
        t=df_results[df_results['id']==narr]['description'].iloc[0]
    except:
        t='Error al interactuar con la API'
    if 'Error al interactuar con la API' in t:
        narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
        narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
        prompt=get_prompt_narr(narr_title)
        # print(prompt)
        result=interact_with_model(prompt)
        print(result)
        new_row = pd.DataFrame([{'id':narr, 'description': result}])
        df_results = pd.concat([df_results, new_row], ignore_index=True)

    

  0%|          | 0/96 [00:00<?, ?it/s]Device set to use cuda:0


  1%|          | 1/96 [00:06<10:27,  6.61s/it]Device set to use cuda:0



Description: This subnarrative, titled "Ukraine is the aggressor," presents a perspective that emphasizes Ukraine's role as the primary instigator of the current conflict. It argues against the notion of Russia as the aggressor, instead attributing the onset of war to Ukraine's actions or policies. This perspective may downplay or deny Russia's military intervention, invasion, or the displacement of civilians as a result, focusing instead on Ukraine's alleged aggressive behavior or transgressions that supposedly precipitated the conflict. The subnarrative aims to shift blame from Russia to Ukraine, potentially to support a specific political stance, agenda, or narrative.

By providing this description, readers can better identify and interpret this subnarrative, recognizing it as an assertion of blame directed at Ukraine rather than Russia, and understanding its potential political implications or biases.


  2%|▏         | 2/96 [00:16<12:56,  8.26s/it]Device set to use cuda:0



Description: This subnarrative, titled "The West are the aggressors," presents a perspective that critically examines the root causes of conflict, specifically highlighting a narrative where the Western nations are depicted as the instigators of war, rather than simply the participants. It challenges the conventional narrative by focusing on the historical context, geopolitical interests, and military interventions of Western powers, often implicating their role in instigating or escalating tensions leading to armed conflict. This subnarrative aims to provoke thought on the complexities of international relations, questioning the simplistic "us vs. them" mentality and urging readers to consider the broader geopolitical implications and the responsibility of powerful nations in the escalation of global tensions.

This carefully crafted description clarifies the subnarrative's focus on attributing blame, not just for the occurrence of war, but for its initiation and perpetuation by West

...

Response: 

This compelling narrative delves into the intricate world of clandestine machinations orchestrated by influential and often unseen forces. It unveils the murky undercurrents of power, where secretive groups wield considerable sway, their actions often hidden from public scrutiny. The piece promises to shed light on covert strategies, maneuvers, and conspiracies that could be reshaping societal, political, or economic landscapes. It invites readers to question the true drivers of change and the shadowy puppeteers pulling the strings behind the veil of transparency in governance and business. This narrative is crucial for news segments that focus on uncovering power dynamics, transparency issues, and the potential impact of unseen forces on everyday lives.


 95%|█████████▌| 20/21 [01:31<00:05,  5.48s/it]Device set to use cuda:0


The narrative "Amplifying Climate Fears" explores the intensifying global concerns over environmental degradation, primarily focusing on the escalating anxieties surrounding climate change. It delves into how various stakeholders—ranging from climate scientists and activists to policymakers and the general public—are experiencing heightened fears due to the mounting evidence of extreme weather events, rising sea levels, and the devastating impacts on biodiversity. The piece highlights the role of media and social platforms in potentially exaggerating these fears, as well as the consequent societal and political ramifications. It underscores the delicate balance between informing the public and avoiding unnecessary alarm, advocating for responsible reporting that fosters informed discussion and action on climate change.

This description aims to clarify that "Amplifying Climate Fears" is not a narrative about an event or a character but rather a discourse on the evolving public percepti

100%|██████████| 21/21 [01:36<00:00,  4.58s/it]

Description:

This narrative explores the intricate intersection of environmental policy, international relations, and geopolitical power dynamics. It delves into how nations are increasingly utilizing green policies—such as renewable energy initiatives, sustainable agriculture practices, and carbon emission reduction strategies—as strategic tools to influence global politics and economic landscapes. Through in-depth analysis, it examines how these environmental measures are not only addressing ecological concerns but also serving as a means to foster economic competitiveness, secure energy independence, and reshape international alliances. This narrative underscores the burgeoning role of green policies in shaping modern geopolitical dynamics, highlighting their multifaceted impact on global affairs and power structures.


In [10]:
df_results

,id,description
0,URW1.1,"\nDescription: This subnarrative, titled ""Ukra..."
1,URW1.2,"\nDescription: This subnarrative, titled ""The ..."
2,URW2.1,"Description:\n\nThis subnarrative, titled ""Rew..."
3,URW2.2,Description: This subnarrative focuses on the ...
4,URW2.3,Description: This subnarrative focuses on effo...
...,...,...
90,CC6.,Description: This narrative delves into the re...
91,CC7.,"Response:\n\nIn this narrative, readers will d..."
92,CC8.,Response: \n\nThis compelling narrative delves...
93,CC9.,"The narrative ""Amplifying Climate Fears"" explo..."


In [11]:
df_results.to_pickle(f'/data/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}_v2.pkl')
df_results.to_json(f'/data/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}_v2.json', orient='records', indent=2, force_ascii=False)

# Improved Description from title and Description

In [12]:
def get_prompt_subnarr(narr, title_subnarr, desc_subnarr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a subnarrative about the narrative "{narr}". Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the subnarrative. Your description will have to help improve the identification of subnarratives in news.

Subnarrative title: {title_subnarr} 
Description: {desc_subnarr}

Task: Generate a description of the subnarrative. 
    """
    return user_prompt

def get_prompt_narr(narr, desc_narr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a narrative. Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the narrative. Your description will have to help improve the identification of narratives in news.

Narrative title: {narr} 
Description: {desc_narr}

Task: Generate a description of the narrative. 
    """
    return user_prompt

In [13]:
name=f'{model_name}_z_s_desc_title_description'

try:
    df_results=pd.read_json(f'/data/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}.json')
    df_results=df_results[df_results['description'].apply(lambda x: 'Error al interactuar con la API' not in x)]
except:    
    df_results=pd.DataFrame(columns=['id', 'description'])

for subnarr in tqdm(allowed_subnarratives):
    if 'OTH' in subnarr:
        continue
    try:
        t=df_results[df_results['id']==subnarr]['description'].iloc[0]
    except:
        t='Error al interactuar con la API'
    if 'Error al interactuar con la API' in t:
        narr=subnarr.split('.')[0]+'.'
        narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
        narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
        subnarr_title=df_titles_desc[df_titles_desc['id']==subnarr]['title'].iloc[0]
        subnarr_desc=df_titles_desc[df_titles_desc['id']==subnarr]['description'].iloc[0]
        prompt=get_prompt_subnarr(narr_title, subnarr_title, subnarr_desc)
        result=interact_with_model(prompt)
        print(result)
        new_row = pd.DataFrame([{'id':subnarr, 'description': result}])
        df_results = pd.concat([df_results, new_row], ignore_index=True)

for narr in tqdm(allowed_narratives):
    try:
        t=df_results[df_results['id']==narr]['description'].iloc[0]
    except:
        t='Error al interactuar con la API'
    if 'Error al interactuar con la API' in t:
        narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
        narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]

        prompt=get_prompt_narr(narr_title, narr_desc)
        result=interact_with_model(prompt)
        print(result)
        new_row = pd.DataFrame([{'id':narr, 'description': result}])
        df_results = pd.concat([df_results, new_row], ignore_index=True)

  1%|          | 1/96 [00:03<05:57,  3.76s/it]Device set to use cuda:0


🔹 Subnarrative Title: Ukraine is the aggressor
🔹 Description: This subnarrative argues against the conventional narrative of Russia as the aggressor, instead positing Ukraine as the initiator of conflict. It asserts that Ukraine provoked the war by allegedly harassing civilians in the Donbass region, thereby attempting to divert blame away from Russia's invasion. This narrative subtly reframes the narrative, emphasizing Ukraine's role in the escalation of hostilities, while disregarding Russia's direct military intervention.


  2%|▏         | 2/96 [00:08<07:02,  4.49s/it]Device set to use cuda:0


🤖

This subnarrative focuses on narratives that attribute the origins and escalation of a conflict solely or predominantly to the actions and intentions of Western entities, such as the United States or the European Union. It encapsulates a perspective that disputes the notion of the conflict being a spontaneous or unprovoked event, instead proposing that it was a calculated response or intentional action by Western powers. This perspective is often underpinned by a claim that these entities deliberately undermined peace processes, like the Minsk II agreement, thereby compelling Russia to take military action, thus portraying them as the primary aggressors. The subnarrative seeks to highlight these direct or indirect actions by Western entities as the catalyst for the conflict, emphasizing a narrative of external blame rather than internal or shared responsibility.


  4%|▍         | 4/96 [00:13<04:56,  3.22s/it]Device set to use cuda:0


...

𝗔𝘀𝗸𝗲𝗻𝗶𝗻𝗴 𝗖𝗼𝗻𝗰𝗲𝗿𝗶𝗼𝗻:

In this compelling narrative, the focus centers on unveiling the clandestine machinations of influential bodies or factions orchestrating a multifaceted strategy to manipulate global climate change discourse. The story delves into the intricate web of covert operations, shadowy dealings, and potential conflicts of interest, suggesting that these powerful entities might be leveraging their influence to steer public opinion and policy-making away from a comprehensive, science-based approach towards climate action. As the narrative unfolds, it promises a revelation of hidden agendas, raising questions about the true motivations behind seemingly coordinated efforts to address one of humanity's most pressing challenges. This piece, aimed at shedding light on the complexities of power dynamics and their impact on climate policy, is a critical exploration of the potential for misinformation and manipulation in the face of urgent global concerns.


 95%|█████████▌| 20/21 [01:29<00:04,  4.72s/it]Device set to use cuda:0


response:

The narrative "Amplifying Climate Fears" delves into the intensification of apprehensions surrounding the impending impacts of climate change. It explores how various entities, such as policymakers, environmental activists, and media outlets, may employ powerful rhetoric to underscore the urgency and severity of ecological disruptions. This narrative scrutinizes the strategic use of alarming statistics, dramatic imagery, and emotionally charged language to evoke fear and spur action or policy changes. It serves as a critical examination of how narratives can shape public perception and influence responses to the climate crisis, emphasizing the delicate balance between raising awareness and potentially fueling widespread anxiety.


100%|██████████| 21/21 [01:33<00:00,  4.46s/it]

🔹 In this narrative, it is posited that environmental policies and initiatives, often referred to as "green policies," are not solely driven by environmental stewardship. Instead, their implementation and promotion by various nations or influential entities are strategically employed as geopolitical tools. The argument suggests that these policies serve as means to exert power, forge alliances, and exert influence in the global arena, rather than being primarily motivated by genuine environmental concerns. This perspective challenges readers to consider the intersection of politics, economics, and environmentalism, prompting a deeper examination of the true motivations behind environmental initiatives on a global scale.


In [14]:
df_results

,id,description
0,URW1.1,🔹 Subnarrative Title: Ukraine is the aggressor...
1,URW1.2,🤖\n\nThis subnarrative focuses on narratives t...
2,URW2.1,"The subnarrative titled ""Rewriting Ukraine’s h..."
3,URW2.2,"The subnarrative titled ""Discrediting Ukrainia..."
4,URW2.3,"The subnarrative titled ""Discrediting Ukrainia..."
...,...,...
90,CC6.,"🤖\n\nThe narrative, titled ""Criticism of the C..."
91,CC7.,Response:\n\nThis narrative will delve into t...
92,CC8.,𝗔𝘀𝗸𝗲𝗻𝗶𝗻𝗴 𝗖𝗼𝗻𝗰𝗲𝗿𝗶𝗼𝗻:\n\nIn this compelling narr...
93,CC9.,"response:\n\nThe narrative ""Amplifying Climate..."


In [15]:
name=f'{model_name}_z_s_desc_title_description'
df_results.to_pickle(f'/data/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}_v2.pkl')
df_results.to_json(f'/data/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}_v2.json', orient='records', indent=2, force_ascii=False)

# Complete Description

In [16]:
def get_prompt_subnarr(narr, title_subnarr, desc_subnarr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a subnarrative about the narrative "{narr}". Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the subnarrative. Your description will have to help improve the identification of subnarratives in news.
Write the characteristics that the model must have in terms of the social, political and economic context, the main actors involved in the subnarrative, what emotions the message seeks to provoke and the conclusions it wants the reader to reach, and the main textual characteristics. 

Subnarrative title: {title_subnarr} 
Description: {desc_subnarr}

Task: Generate a description of the subnarrative. 
    """
    return user_prompt

def get_prompt_narr(narr, desc_narr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a narrative. Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the narrative. Your description will have to help improve the identification of narratives in news.
Write the characteristics that the model must have in terms of the social, political and economic context, the main actors involved in the narrative, what emotions the message seeks to provoke and the conclusions it wants the reader to reach, and the main textual characteristics. 

Narrative title: {narr} 
Description: {desc_narr}

Task: Generate a description of the subnarrative. 
    """
    return user_prompt

In [17]:
name=f'{model_name}_z_s_desc_title_description_complete'

try:
    df_results=pd.read_json(f'/data/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}.json')
    df_results=df_results[df_results['description'].apply(lambda x: 'Error al interactuar con la API' not in x)]
except:    
    df_results=pd.DataFrame(columns=['id', 'description'])

for subnarr in tqdm(allowed_subnarratives):
    if 'OTH' in subnarr:
        continue
    try:
        t=df_results[df_results['id']==subnarr]['description'].iloc[0]
    except:
        t='Error al interactuar con la API'
    if 'Error al interactuar con la API' in t:
        narr=subnarr.split('.')[0]+'.'
        narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
        narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
        subnarr_title=df_titles_desc[df_titles_desc['id']==subnarr]['title'].iloc[0]
        subnarr_desc=df_titles_desc[df_titles_desc['id']==subnarr]['description'].iloc[0]
        prompt=get_prompt_subnarr(narr_title, subnarr_title, subnarr_desc)
        result=interact_with_model(prompt)
        print(result)
        new_row = pd.DataFrame([{'id':subnarr, 'description': result}])
        df_results = pd.concat([df_results, new_row], ignore_index=True)

for narr in tqdm(allowed_narratives):
    try:
        t=df_results[df_results['id']==narr]['description'].iloc[0]
    except:
        t='Error al interactuar con la API'
    if 'Error al interactuar con la API' in t:
        narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
        narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]

        prompt=get_prompt_narr(narr_title, narr_desc)
        result=interact_with_model(prompt)
        print(result)
        new_row = pd.DataFrame([{'id':narr, 'description': result}])
        df_results = pd.concat([df_results, new_row], ignore_index=True)



  1%|          | 1/96 [00:07<12:20,  7.79s/it]Device set to use cuda:0


Response:

This subnarrative, titled "Ukraine is the aggressor," centers around a narrative shift that absolves Russia of responsibility for the ongoing conflict. It portrays Ukraine as the instigator, employing the tactic of "blame shifting." The social and political context is characterized by a geopolitical power struggle between Ukraine and Russia, with economic implications including potential sanctions and influence over global energy markets. The main actors are Russia, represented as the defender, and Ukraine, portrayed as the offender. The emotional undertone is antagonistic, fostering a sense of suspicion towards Ukraine and sympathy for Russia's purported defensive stance. The conclusion the author seeks to instill is that Russia is not the initiator of the conflict but is instead responding to Ukraine's alleged transgressions. The textual characteristics include the use of loaded language ("secretly provoked," "harassing"), omission of context (specific details about Russia

  2%|▏         | 2/96 [00:16<12:40,  8.09s/it]Device set to use cuda:0


The subnarrative "The West are the aggressors" focuses on a narrative that re-frames the conflict as a direct outcome of Western interventions, rather than a response to Russian or other non-Western actions. This perspective centers on the notion that Western powers, such as the United States and the European Union, are primarily responsible for the escalation of tensions and the ensuing crisis. The language used in this subnarrative often employs direct or implicating statements that attribute the conflict's origins to the West's actions, such as sabotaging peace agreements or imposing sanctions that inadvertently propel the antagonist nation (Russia) into a defensive posture. The emotional undertone seeks to evoke feelings of resentment, mistrust, and potentially solidarity among the audience with those portrayed as victims of Western aggression. The conclusion this subnarrative aims to draw is that the West, through their policies and actions, have instigated a situation that necess

  4%|▍         | 4/96 [00:28<10:37,  6.93s/it]Device set to use cuda:0


...

Answer:

The subnarrative, nestled within the broader discourse on climate change, delves into the shadows of power, where unseen forces allegedly manipulate global environmental policies. It portrays a society grappling with the looming crisis of climate change, where the stage is set for a dramatic confrontation between the common people, represented by grassroots movements and concerned environmentalists, and formidable forces, embodied by influential corporations, political elites, and possibly even international organizations. The narrative stirs the emotions of skepticism, outrage, and fear, as it uncovers allegations of clandestine agreements and covert operations designed to obfuscate climate action. It seeks to provoke the reader into questioning the true intentions behind policy decisions, demanding greater transparency and accountability from those in power. The narrative's conclusion invites the audience to consider the weight of collective responsibility in combating clima

 95%|█████████▌| 20/21 [02:55<00:10, 10.14s/it]Device set to use cuda:0


Response:

The subnarrative, "Amplifying Climate Fears," delves into a potent discourse that amplifies the perceived gravity and repercussions of climate change. It operates within a stark social, political, and economic context, characterized by escalating environmental concerns, heightened global political tensions over climate policy, and the impending economic shifts driven by transitioning to sustainable practices. The main actors in this narrative encompass environmental activists, political leaders, multinational corporations, and the general public, each playing distinctive roles in either exacerbating or mitigating the climate crisis.

This subnarrative seeks to provoke a range of intense emotions—anxiety, urgency, and perhaps even a sense of moral outrage—among the readership, as it underscores the potential for catastrophic environmental and socio-economic consequences if current trends persist. It aims to galvanize public opinion, pressuring policymakers to enact stringent 

100%|██████████| 21/21 [03:08<00:00,  8.99s/it]

❗ This description will serve as a guide for enhancing the readability and interpretation of the narrative for news articles.

The subnarrative "Green policies are geopolitical instruments" touches upon the intricate relationship between environmental concerns and global power dynamics. It argues that environmental policies, often touted as 'green initiatives,' are not merely driven by a genuine concern for the planet but are increasingly seen as strategic tools wielded by nations to assert influence, negotiate alliances, and reshape global geopolitics.

Key characteristics to highlight in the description:

1. **Social Context**:
   - Emerging awareness of environmental issues globally.
   - Growing skepticism towards traditional environmental motives and motivations.

2. **Political Context**:
   - The escalating role of environmental policies in international relations.
   - Geopolitical maneuvering through 'green' agendas.
   - Potential for conflict or cooperation arising from diff

In [18]:
df_results

,id,description
0,URW1.1,"Response:\n\nThis subnarrative, titled ""Ukrain..."
1,URW1.2,"The subnarrative ""The West are the aggressors""..."
2,URW2.1,"RESPONSE:\n\nThis subnarrative, titled ""Rewrit..."
3,URW2.2,"🤖\n\nThe subnarrative, titled ""Discrediting Uk..."
4,URW2.3,RESPONSE:\n\nThis subnarrative operates within...
...,...,...
90,CC6.,The narrative will revolve around a contentiou...
91,CC7.,👾 Description:\n\nThis subnarrative delves int...
92,CC8.,"Answer:\n\nThe subnarrative, nestled within th..."
93,CC9.,"Response:\n\nThe subnarrative, ""Amplifying Cli..."


In [19]:
name=f'{model_name}_z_s_desc_title_description_complete'
df_results.to_pickle(f'/data/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}_v2.pkl')
df_results.to_json(f'/data/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}_v2.json', orient='records', indent=2, force_ascii=False)

In [ ]:
df_titles_desc=pd.read_json('/data/jfraile/Programs/datasets/SemEval_2025_T10_S2/titles_and_descriptions.json')
ids_with_title_desc=df_titles_desc.id.values

allowed_classifications_ids=['URW1.1', 'URW1.2', 'URW1.OTH', 'URW2.1', 'URW2.2', 'URW2.3', 'URW2.4', 'URW2.5', 'URW2.6', 'URW2.7', 'URW2.8', 'URW2.OTH', 'URW3.1', 'URW3.2', 'URW3.3', 'URW3.OTH', 'URW4.1', 'URW4.2', 'URW4.3', 'URW4.4', 'URW4.5', 'URW4.OTH', 'URW5.1', 'URW5.2', 'URW5.3', 'URW5.OTH', 'URW6.1', 'URW6.2', 'URW6.3', 'URW6.OTH', 'URW7.1', 'URW7.2', 'URW7.3', 'URW7.4', 'URW7.5', 'URW7.6', 'URW7.OTH', 'URW8.1', 'URW8.2', 'URW8.OTH', 'URW9.1', 'URW9.2', 'URW9.OTH', 'URW10.1', 'URW10.2', 'URW10.3', 'URW10.4', 'URW10.OTH', 'URW11.OTH', 'CC1.1', 'CC1.2', 'CC1.3', 'CC1.OTH', 'CC2.1', 'CC2.2', 'CC2.3', 'CC2.4', 'CC2.OTH', 'CC3.1', 'CC3.2', 'CC3.OTH', 'CC4.1', 'CC4.2', 'CC4.3', 'CC4.4', 'CC4.5', 'CC4.6', 'CC4.7', 'CC4.8', 'CC4.OTH', 'CC5.1', 'CC5.2', 'CC5.3', 'CC5.4', 'CC5.OTH', 'CC6.1', 'CC6.2', 'CC6.3', 'CC6.OTH', 'CC7.1', 'CC7.2', 'CC7.3', 'CC7.4', 'CC7.OTH', 'CC8.1', 'CC8.2', 'CC8.OTH', 'CC9.1', 'CC9.2', 'CC9.3', 'CC9.4', 'CC9.OTH', 'CC10.1', 'CC10.2', 'CC10.OTH', 'OTH.OTH']

# Description from titles

In [ ]:
def get_prompt(narr, title_subnarr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a subnarrative about the narrative "{narr}". Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the subnarrative. Your description will have to help improve the identification of subnarratives in news.

Subnarrative title: {title_subnarr} 

Task: Generate a description of the subnarrative. 
    """
    return user_prompt

In [ ]:
narr='URW11.'
narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
prompt=get_prompt(narr_title, narr_title)
result=interact_with_model(prompt)

In [ ]:
result

In [ ]:
df_results=pd.DataFrame(columns=['id', 'description'])

for subnarr in tqdm(allowed_classifications_ids):
    if 'OTH' in subnarr:
        continue
    narr=subnarr.split('.')[0]+'.'
    narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
    narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
    subnarr_title=df_titles_desc[df_titles_desc['id']==subnarr]['title'].iloc[0]
    subnarr_desc=df_titles_desc[df_titles_desc['id']==subnarr]['description'].iloc[0]
    prompt=get_prompt(narr_title, subnarr_title)
    result=interact_with_model(prompt)
    new_row = pd.DataFrame([{'id':subnarr, 'description': result}])
    df_results = pd.concat([df_results, new_row], ignore_index=True)


    

In [ ]:
df_results

In [ ]:
name='calme_z_s_desc_only_title'
df_results.to_pickle(f'/data/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}.pkl')

# Improved Description from title and Description

In [ ]:
def get_prompt(narr, title_subnarr, desc_subnarr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a subnarrative about the narrative "{narr}". Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the subnarrative. Your description will have to help improve the identification of subnarratives in news.

Subnarrative title: {title_subnarr} 
Description: {desc_subnarr}

Task: Generate a description of the subnarrative. 
    """
    return user_prompt

In [ ]:
narr='URW11.'
narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
prompt=get_prompt(narr_title, narr_title, narr_desc)
result=interact_with_model(prompt)

In [ ]:
result

In [ ]:
df_results=pd.DataFrame(columns=['id', 'description'])

for subnarr in tqdm(allowed_classifications_ids):
    if 'OTH' in subnarr:
        continue
    narr=subnarr.split('.')[0]+'.'
    narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
    narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
    subnarr_title=df_titles_desc[df_titles_desc['id']==subnarr]['title'].iloc[0]
    subnarr_desc=df_titles_desc[df_titles_desc['id']==subnarr]['description'].iloc[0]
    prompt=get_prompt(narr_title, subnarr_title, subnarr_desc)
    result=interact_with_model(prompt)
    new_row = pd.DataFrame([{'id':subnarr, 'description': result}])
    df_results = pd.concat([df_results, new_row], ignore_index=True)

In [ ]:
df_results

In [ ]:
name='calme_z_s_desc_title_description'
df_results.to_pickle(f'/data/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}.pkl')

# Completed Description

In [ ]:
def get_prompt(narr, title_subnarr, desc_subnarr):

    user_prompt = f"""
You are an expert at improving prompts for language models. You are going to receive the title of a subnarrative about the narrative "{narr}". Your mission will be to write a descriptive paragraph to improve the readability and interpretation of the subnarrative. Your description will have to help improve the identification of subnarratives in news.
Write the characteristics that the model must have in terms of the social, political and economic context, the main actors involved in the subnarrative, what emotions the message seeks to provoke and the conclusions it wants the reader to reach, and the main textual characteristics. 

Subnarrative title: {title_subnarr} 
Description: {desc_subnarr}

Task: Generate a description of the subnarrative. 
    """
    return user_prompt

In [ ]:
narr='URW11.'
narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
prompt=get_prompt(narr_title, narr_title, narr_desc)
result=interact_with_model(prompt)

In [ ]:
result

In [ ]:
df_results=pd.DataFrame(columns=['id', 'description'])

for subnarr in tqdm(allowed_classifications_ids):
    if 'OTH' in subnarr:
        continue
    narr=subnarr.split('.')[0]+'.'
    narr_title=df_titles_desc[df_titles_desc['id']==narr]['title'].iloc[0]
    narr_desc=df_titles_desc[df_titles_desc['id']==narr]['description'].iloc[0]
    subnarr_title=df_titles_desc[df_titles_desc['id']==subnarr]['title'].iloc[0]
    subnarr_desc=df_titles_desc[df_titles_desc['id']==subnarr]['description'].iloc[0]
    prompt=get_prompt(narr_title, subnarr_title, subnarr_desc)
    result=interact_with_model(prompt)
    new_row = pd.DataFrame([{'id':subnarr, 'description': result}])
    df_results = pd.concat([df_results, new_row], ignore_index=True)

In [ ]:
df_results

In [ ]:
name='calme_z_s_desc_title_description_complete'
df_results.to_pickle(f'/data/jfraile/Programs/SemEval_2025_T10_S2/description_generation/zero_shot_desc/{name}.pkl')